# Method 2: BERT Hybrid + XGBoost（正則化 + GPU）

### 相比 Method 1，這個版本改動了什麼？

| 項目 | Method 1 | Method 2 |
|------|----------|----------|
| 設定來源 | 硬編碼 | `config/config.yaml` |
| GPU 支援 | 否（CPU） | 是（SentenceTransformer + XGBoost） |
| XGBoost `n_estimators` | 100 | 300 |
| XGBoost `learning_rate` | 0.1 | 0.05 |
| XGBoost `max_depth` | 4 | 3 |
| XGBoost 正則化 | 無 | `subsample=0.8`, `colsample_bytree=0.8`, `reg_alpha=0.1`, `reg_lambda=1.0` |

**為什麼要加正則化？**  
訓練集只有 2,000 筆，XGBoost 的樹很容易對訓練集死背。加入正則化後：
- `subsample` / `colsample_bytree`：每棵樹只看部分樣本和特徵，強迫多樣性
- `reg_alpha` (L1)：讓不重要的特徵權重歸零
- `reg_lambda` (L2)：縮小所有葉節點的分數，防止極端值
- `max_depth=3` + `learning_rate=0.05`：更多棵矮樹，比少棵高樹更難 overfit

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# ── 專案根目錄（往上找到含有 config/ 的目錄，不依賴 Jupyter 啟動位置）────────
def find_project_root(marker: str = 'config') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"找不到專案根目錄（marker='{marker}'）")

ROOT = find_project_root()
print(f'>> 專案根目錄: {ROOT}')

# ── 載入 config ───────────────────────────────────────────────────────────────
with open(ROOT / 'config' / 'config.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# ── device 解析（auto → cuda / cpu）─────────────────────────────────────────
def resolve_device(value: str) -> str:
    if value == 'auto':
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    return value

DEVICE = resolve_device(cfg['global']['device'])
print(f'>> 使用裝置: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

# ── 載入資料 ──────────────────────────────────────────────────────────────────
train_df   = pd.read_csv(ROOT / cfg['paths']['train'])
test_df    = pd.read_csv(ROOT / cfg['paths']['test'])
sample_sub = pd.read_csv(ROOT / cfg['paths']['sample_submission'])

print(f'Train: {train_df.shape} / Test: {test_df.shape}')

In [2]:
# ── 前處理
pre = cfg['preprocessing']

def clean_for_tfidf(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = text.replace(pre['track_a']['lrb_token'], ' ').replace(pre['track_a']['rrb_token'], ' ')
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_for_bert(text):
    if not isinstance(text, str):
        return ''
    text = text.replace(pre['track_a']['lrb_token'], pre['track_b']['lrb_replace'])
    text = text.replace(pre['track_a']['rrb_token'], pre['track_b']['rrb_replace'])
    return text.strip()

def extract_meta_features(df):
    meta = {}
    if cfg['features']['meta']['use_question_mark']:
        meta['q_mark_count'] = df['TEXT'].apply(lambda x: str(x).count('?'))
    if cfg['features']['meta']['use_exclaim_mark']:
        meta['e_mark_count'] = df['TEXT'].apply(lambda x: str(x).count('!'))
    return pd.DataFrame(meta).values

print('>> 前處理中...')
train_text_tfidf = train_df['TEXT'].apply(clean_for_tfidf)
test_text_tfidf  = test_df['TEXT'].apply(clean_for_tfidf)
train_text_bert  = train_df['TEXT'].apply(clean_for_bert)
test_text_bert   = test_df['TEXT'].apply(clean_for_bert)
print('>> 前處理完成')

>> 前處理中...
>> 前處理完成


In [3]:
# ── TF-IDF + LDA Soft Topics
tfidf_cfg = cfg['features']['tfidf']
lda_cfg   = cfg['features']['lda']

print('>> 建立 TF-IDF 矩陣...')
tfidf = TfidfVectorizer(
    max_features = tfidf_cfg['max_features'],
    stop_words   = tfidf_cfg['stop_words'],
    min_df       = tfidf_cfg['min_df'],
)
train_tfidf = tfidf.fit_transform(train_text_tfidf)
test_tfidf  = tfidf.transform(test_text_tfidf)

print('>> 訓練 LDA Soft Topics...')
lda = LatentDirichletAllocation(
    n_components = lda_cfg['n_topics'],
    random_state = lda_cfg['random_state'],
)
X_train_topics = lda.fit_transform(train_tfidf)
X_test_topics  = lda.transform(test_tfidf)

train_meta = extract_meta_features(train_df)
test_meta  = extract_meta_features(test_df)

print(f'   Topics shape: {X_train_topics.shape} / Meta shape: {train_meta.shape}')

>> 建立 TF-IDF 矩陣...
>> 訓練 LDA Soft Topics...
   Topics shape: (2000, 3) / Meta shape: (2000, 2)


In [4]:
# ── Sentence Transformer Embeddings（GPU 加速）
st_cfg = cfg['features']['sentence_transformer']
st_device = resolve_device(st_cfg['device'])

print(f'>> 載入 Sentence Transformer: {st_cfg["model_name"]}  (device={st_device})')
st_model = SentenceTransformer(st_cfg['model_name'], device=st_device)

print('>> 生成 Train Embeddings...')
X_train_bert = st_model.encode(
    train_text_bert.tolist(),
    batch_size        = st_cfg['batch_size'],
    show_progress_bar = True,
)
print('>> 生成 Test Embeddings...')
X_test_bert = st_model.encode(
    test_text_bert.tolist(),
    batch_size        = st_cfg['batch_size'],
    show_progress_bar = True,
)
print(f'   Embeddings shape: {X_train_bert.shape}')

>> 載入 Sentence Transformer: all-MiniLM-L6-v2  (device=cuda)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

>> 生成 Train Embeddings...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

>> 生成 Test Embeddings...


Batches:   0%|          | 0/344 [00:00<?, ?it/s]

   Embeddings shape: (2000, 384)


In [5]:
# ── 特徵拼接
X_train_final = np.hstack([X_train_bert, X_train_topics, train_meta])
X_test_final  = np.hstack([X_test_bert,  X_test_topics,  test_meta])
y_train = train_df['LABEL'].values

emb_dim = X_train_bert.shape[1]
print(f'最終特徵維度: {X_train_final.shape}')
print(f'  [BERT {emb_dim}] + [LDA {X_train_topics.shape[1]}] + [Meta {train_meta.shape[1]}]')

最終特徵維度: (2000, 389)
  [BERT 384] + [LDA 3] + [Meta 2]


In [ ]:
import json
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_validate

# ── XGBoost（正則化 + GPU）────────────────────────────────────────────────────
xgb_cfg    = cfg['models']['xgboost']
xgb_device = xgb_cfg['device']   # 'cuda' 或 'cpu'；非 auto，不需 resolve

print('====== [ Method 2: XGBoost + 正則化 ] ======')
print(f'  n_estimators={xgb_cfg["n_estimators"]}, lr={xgb_cfg["learning_rate"]}, max_depth={xgb_cfg["max_depth"]}')
print(f'  subsample={xgb_cfg["subsample"]}, colsample_bytree={xgb_cfg["colsample_bytree"]}')
print(f'  reg_alpha={xgb_cfg["reg_alpha"]}, reg_lambda={xgb_cfg["reg_lambda"]}')
print(f'  device={xgb_device}  (device=cuda 時不設 n_jobs)')

xgb = XGBClassifier(
    n_estimators     = xgb_cfg['n_estimators'],
    learning_rate    = xgb_cfg['learning_rate'],
    max_depth        = xgb_cfg['max_depth'],
    random_state     = xgb_cfg['random_state'],
    eval_metric      = xgb_cfg['eval_metric'],
    device           = xgb_device,
    subsample        = xgb_cfg['subsample'],
    colsample_bytree = xgb_cfg['colsample_bytree'],
    reg_alpha        = xgb_cfg['reg_alpha'],
    reg_lambda       = xgb_cfg['reg_lambda'],
    # n_jobs 在 device=cuda 下由 CUDA 控制，不設定以避免衝突
)

cv_cfg  = cfg['evaluation']['cross_validation']
scoring = {
    'accuracy':  'accuracy',
    'precision': make_scorer(precision_score, average='macro', zero_division=0),
    'recall':    make_scorer(recall_score,    average='macro', zero_division=0),
    'f1':        make_scorer(f1_score,        average='macro', zero_division=0),
}

cv_results = cross_validate(
    xgb, X_train_final, y_train,
    cv=cv_cfg['cv'], scoring=scoring, return_train_score=False,
)

acc_mean  = cv_results['test_accuracy'].mean()
acc_std   = cv_results['test_accuracy'].std()
prec_mean = cv_results['test_precision'].mean()
rec_mean  = cv_results['test_recall'].mean()
f1_mean   = cv_results['test_f1'].mean()

print(f'\n{"指標":<12} {"平均":>8}')
print('-' * 22)
print(f'{"Accuracy":<12} {acc_mean:>8.4f}  (std={acc_std:.4f})')
print(f'{"Precision":<12} {prec_mean:>8.4f}')
print(f'{"Recall":<12} {rec_mean:>8.4f}')
print(f'{"F1":<12} {f1_mean:>8.4f}')
print(f'\n（Baseline exp_001: Accuracy=0.6485）')

# ── 寫入 experiments/exp_003/metrics.json ────────────────────────────────────
exp_dir = ROOT / 'experiments' / 'exp_003_method2_regularized'
metrics = {
    'experiment_id':    'exp_003',
    'cv_folds':          cv_cfg['cv'],
    'cv_accuracy':       cv_results['test_accuracy'].tolist(),
    'cv_accuracy_mean':  round(acc_mean,  4),
    'cv_accuracy_std':   round(acc_std,   4),
    'cv_precision_mean': round(prec_mean, 4),
    'cv_recall_mean':    round(rec_mean,  4),
    'cv_f1_mean':        round(f1_mean,   4),
    'kaggle_score':      None,
    'note':              '',
}
with open(exp_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f'\n✅ metrics 已寫入: {exp_dir / "metrics.json"}')

In [ ]:
# ── 訓練全量模型並輸出 Submission
xgb.fit(X_train_final, y_train)
test_preds = xgb.predict(X_test_final)

out_path = ROOT / cfg['paths']['submissions']['method2']
out_path.parent.mkdir(parents=True, exist_ok=True)
submission = pd.DataFrame({'row_id': test_df['row_id'], 'LABEL': test_preds})
submission.to_csv(out_path, index=False)
print(f'✅ 已儲存: {out_path}')